📘 1 — Imports and Drive download

In [13]:
# 🧠 1. Imports
import os, gdown, pandas as pd, numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
import matplotlib.pyplot as plt
import seaborn as sns

file_ids = [
    "1YFHIeCQOMDoWgYdS71reBxUi_0ee-QTF",
    "187dmxycOB1iSBW95XCkC42gLYZOnoOib",
    "1uu4G9hHXgcjpQucFewhazyq3K9U7Pykz",
    "11kr3mjS3sROYIKG0RtLCtNOzDTi6vDXk",
    "1EKchutqTAQzhrsgR5oZzbFGT3HFl0Z1X",
    "1jPY0etKd-yl20OI-0mT_RWaZnyJNY9j0",
    "1jG0hNoQbZoXTeIpL8l4OLV_H1MAUchZ2",
    "188Mb-fYJKT5BGxwsoNc2G3xiDOyVAhye",
]

os.makedirs("/content/data", exist_ok=True)
csv_paths = []

for fid in file_ids:
    url = f"https://drive.google.com/uc?id={fid}"
    out_path = f"/content/data/{fid}.csv"
    gdown.download(url, out_path, quiet=False)
    csv_paths.append(out_path)

dfs = [pd.read_csv(p) for p in csv_paths]
df = pd.concat(dfs, ignore_index=True)
print("Merged shape:", df.shape)

Downloading...
From (original): https://drive.google.com/uc?id=1YFHIeCQOMDoWgYdS71reBxUi_0ee-QTF
From (redirected): https://drive.google.com/uc?id=1YFHIeCQOMDoWgYdS71reBxUi_0ee-QTF&confirm=t&uuid=b81a610c-ac75-4fde-8b5f-8384214a4d40
To: c:\content\data\1YFHIeCQOMDoWgYdS71reBxUi_0ee-QTF.csv













































100%|██████████| 225M/225M [00:07<00:00, 29.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=187dmxycOB1iSBW95XCkC42gLYZOnoOib
From (redirected): https://drive.google.com/uc?id=187dmxycOB1iSBW95XCkC42gLYZOnoOib&confirm=t&uuid=e5d57880-3c08-427a-84d6-0e16d90f6dd0
To: c:\content\data\187dmxycOB1iSBW95XCkC42gLYZOnoOib.csv































100%|██████████| 135M/135M [00:06<00:00, 21.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uu4G9hHXgcjpQucFewhazyq3K9U7Pykz
To: c:\content\data\1uu4G9hHXgcjpQucFewhazyq3K9U7Pykz.csv

























100%|██████████| 52.0M/52.0M [00:05<00:00, 9.81MB/s]
Downloading...
Fro

Merged shape: (2830743, 79)


📘 2 — Data Cleaning and Label Grouping

In [14]:
# Drop duplicates, NaN, Inf, low-variance features
df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

num_cols = df.select_dtypes(include=[np.number]).columns
low_var = df[num_cols].std()[df[num_cols].std() < 0.01].index
df = df.drop(columns=low_var)
print("Dropped low-variance cols:", list(low_var))
print("Cleaned shape:", df.shape)

def group_label(lbl):
    l = str(lbl).lower()
    if 'benign' in l: return 'Benign'
    if any(x in l for x in ['dos','ddos','hulk','goldeneye','slowloris','slowhttptest']): return 'Dos/DDoS'
    if 'portscan' in l: return 'PortScan'
    if any(x in l for x in ['patator','brute','ftp','ssh']): return 'Brute Force'
    if any(x in l for x in ['web','xss','sql','injection']): return 'Web Attack'
    if 'bot' in l: return 'Botnet ARES'
    return 'Other'

df['label_group'] = df[' Label'].apply(group_label)
df = df[df['label_group'] != 'Other']
print(df['label_group'].value_counts())


Dropped low-variance cols: [' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' CWE Flag Count', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Cleaned shape: (2520798, 69)
label_group
Benign         2095057
Dos/DDoS        321759
PortScan         90694
Brute Force      10620
Botnet ARES       1948
Web Attack         673
Name: count, dtype: int64


📘 3 — Label Encoding, Feature Selection, and Deduplication

In [15]:
paper_order = ['Benign','Botnet ARES','Brute Force','Dos/DDoS','PortScan','Web Attack']
paper_code  = {cls:i for i,cls in enumerate(paper_order)}
df = df[df['label_group'].isin(paper_order)]
df['y'] = df['label_group'].map(paper_code)
print("Label mapping:", paper_code)

drop_cols = [' Label','label_group']
df_features = df.drop(columns=drop_cols, errors='ignore')
for c in ['Flow ID','Source IP','Destination IP','Timestamp','Unnamed: 0']:
    if c in df_features.columns: df_features = df_features.drop(columns=c)
df_features = pd.get_dummies(df_features, drop_first=True)
print("Feature matrix shape:", df_features.shape)
# Drop duplicates based on feature values (ignore indices)
df_features['y'] = df['y'].values
df_unique = df_features.drop_duplicates(subset=df_features.columns[:-1])
print("Shape after removing near-duplicates:", df_unique.shape)

# Separate features & labels
X_unique = df_unique.drop(columns='y')
y_unique = df_unique['y']


Label mapping: {'Benign': 0, 'Botnet ARES': 1, 'Brute Force': 2, 'Dos/DDoS': 3, 'PortScan': 4, 'Web Attack': 5}
Feature matrix shape: (2520751, 69)
Shape after removing near-duplicates: (2520054, 69)


📘 4 — Train, Validation, and Test Split

In [16]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_unique, y_unique, test_size=0.2, stratify=y_unique, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
)

print("Counts: train, val, test =", Counter(y_train), Counter(y_val), Counter(y_test))

Counts: train, val, test = Counter({0: 1508351, 3: 231661, 4: 64894, 2: 7646, 1: 1402, 5: 484}) Counter({0: 167595, 3: 25740, 4: 7210, 2: 850, 1: 156, 5: 54}) Counter({0: 418986, 3: 64350, 4: 18026, 2: 2124, 1: 390, 5: 135})


📘 5 — Feature Scaling using StandardScaler

In [17]:
# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

📘 6 — Class Balancing: Undersampling + SMOTE and Computing Class Weights

In [18]:
# Safe undersample majority class (Benign)
majority_class = 0
rus = RandomUnderSampler(
    sampling_strategy={majority_class: int(0.5 * (y_train == majority_class).sum())},
    random_state=42
)
X_rus, y_rus = rus.fit_resample(X_train_scaled.values, y_train.values)

# SMOTE to balance minority classes
smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_rus, y_rus)

print("Class distribution after balancing:", Counter(y_bal))

# Compute class weights
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_bal),
    y=y_bal
)
class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)

Class distribution after balancing: Counter({0: 754175, 1: 754175, 2: 754175, 3: 754175, 4: 754175, 5: 754175})
Class weights: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0}


📘 7 — Neural Network Model Definition: DNN with Gaussian Noise, BatchNorm, Dropout, and L2 Regularization

In [19]:
num_features = X_bal.shape[1]
num_classes  = len(np.unique(y_bal))

tf.keras.utils.set_random_seed(42)

inputs = keras.Input(shape=(num_features,))
x = keras.layers.GaussianNoise(1e-2)(inputs)
x = keras.layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.4)(x)

for units in [256,128,64,32]:
    x = keras.layers.Dense(units, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.4)(x)

outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 68)]              0         
                                                                 
 gaussian_noise (GaussianNo  (None, 68)                0         
 ise)                                                            
                                                                 
 dense (Dense)               (None, 512)               35328     
                                                                 
 batch_normalization (Batch  (None, 512)               2048      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 256)               131328

📘 8 — Model Training with Early Stopping, LR Reduction, and Class Weights

In [20]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, monitor='val_loss')
]

history = model.fit(
    X_bal, y_bal,
    validation_data=(X_test_scaled.values, y_test.values),
    epochs=60,
    batch_size=256,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2
)

Epoch 1/60
17676/17676 - 129s - loss: 0.2079 - accuracy: 0.9508 - val_loss: 0.2123 - val_accuracy: 0.9360 - lr: 0.0010 - 129s/epoch - 7ms/step
Epoch 2/60
17676/17676 - 135s - loss: 0.1727 - accuracy: 0.9575 - val_loss: 0.2302 - val_accuracy: 0.9314 - lr: 0.0010 - 135s/epoch - 8ms/step
Epoch 3/60
17676/17676 - 134s - loss: 0.1669 - accuracy: 0.9584 - val_loss: 0.1913 - val_accuracy: 0.9407 - lr: 0.0010 - 134s/epoch - 8ms/step
Epoch 4/60
17676/17676 - 135s - loss: 0.1659 - accuracy: 0.9580 - val_loss: 0.2031 - val_accuracy: 0.9426 - lr: 0.0010 - 135s/epoch - 8ms/step
Epoch 5/60
17676/17676 - 135s - loss: 0.1632 - accuracy: 0.9586 - val_loss: 0.2029 - val_accuracy: 0.9418 - lr: 0.0010 - 135s/epoch - 8ms/step
Epoch 6/60
17676/17676 - 137s - loss: 0.1620 - accuracy: 0.9590 - val_loss: 0.2099 - val_accuracy: 0.9435 - lr: 0.0010 - 137s/epoch - 8ms/step
Epoch 7/60
17676/17676 - 136s - loss: 0.1444 - accuracy: 0.9621 - val_loss: 0.1878 - val_accuracy: 0.9461 - lr: 5.0000e-04 - 136s/epoch - 8ms/

📘 9 — Model Evaluation: Predictions, Classification Report, and Confusion Matrix

In [ ]:
# --- Predictions ---
y_pred = np.argmax(model.predict(X_test_scaled, batch_size=1024), axis=1)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
print("\n--- Confusion Matrix ---")
print(cm)

# Save to model_files folder
output_path = "model_files/confusion_matrix.txt"
np.savetxt(output_path, cm, fmt="%d")
print(f"📄 Confusion matrix saved as: {output_path}")

# --- Confusion Matrix Plot ---
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()

# Add labels
tick_marks = np.arange(len(np.unique(y_test)))
plt.xticks(tick_marks, np.unique(y_test), rotation=45)
plt.yticks(tick_marks, np.unique(y_test))

plt.ylabel("True Label")
plt.xlabel("Predicted Label")

plt.tight_layout()
plt.show()

: 

📘 10 — Save Trained Model and Preprocessing Artefacts

In [ ]:
model.save("model_files/dnn_cicids2017_model.h5")
joblib.dump(scaler, "model_files/scaler.gz")
joblib.dump(paper_code, "model_files/label_mapping.pkl")
print("Saved model and artefacts in model_files/")

Saved model and artefacts in current folder


c:\Users\hahvi\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
